# Évaluation finale sur l'ensemble de test

Premier et unique usage de X_test/y_test dans tout le pipeline.
Ne JAMAIS relancer d'optimisation après ce script en se basant sur ces
résultats -- ce serait re-introduire une fuite de données.

Auteur : Rasmané

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, precision_recall_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DOSSIER = r"C:\Users\hp\Documents\Fraude_detection\data"
DOSSIER_SORTIE = r"C:\Users\hp\Documents\Fraude_detection\data"

## 0. CHARGEMENT DU MODÈLE FINAL ET DES DONNÉES DE TEST

In [ ]:
modele = joblib.load(f"{DOSSIER}/modele_xgboost_optimise.joblib")

X_test = pd.read_csv(f"{DOSSIER}/X_test.csv")
y_test = pd.read_csv(f"{DOSSIER}/y_test.csv").squeeze()

# On recharge aussi les résultats de validation pour comparaison
X_val = pd.read_csv(f"{DOSSIER}/X_val.csv")
y_val = pd.read_csv(f"{DOSSIER}/y_val.csv").squeeze()

print(f"X_test : {X_test.shape} | Taux de fraude : {y_test.mean()*100:.4f}%")
print("Ce fichier n'a été utilisé pour AUCUNE décision jusqu'à présent.")

## 1. PRÉDICTIONS SUR VALIDATION ET SUR TEST

In [ ]:
proba_val = modele.predict_proba(X_val)[:, 1]
proba_test = modele.predict_proba(X_test)[:, 1]

pred_val = (proba_val >= 0.5).astype(int)
pred_test = (proba_test >= 0.5).astype(int)

## 2. COMPARAISON VALIDATION vs TEST (contrôle de cohérence)

In [ ]:
print("\n" + "=" * 70)
print("COMPARAISON VALIDATION vs TEST (contrôle de généralisation)")
print("=" * 70)

tableau_comparaison = pd.DataFrame({
    "métrique": ["AUC-PR", "AUC-ROC", "F1-score", "Précision", "Rappel"],
    "validation": [
        average_precision_score(y_val, proba_val),
        roc_auc_score(y_val, proba_val),
        f1_score(y_val, pred_val),
        precision_score(y_val, pred_val, zero_division=0),
        recall_score(y_val, pred_val),
    ],
    "test": [
        average_precision_score(y_test, proba_test),
        roc_auc_score(y_test, proba_test),
        f1_score(y_test, pred_test),
        precision_score(y_test, pred_test, zero_division=0),
        recall_score(y_test, pred_test),
    ],
})
tableau_comparaison["écart"] = tableau_comparaison["test"] - tableau_comparaison["validation"]
print(tableau_comparaison.round(4).to_string(index=False))

print("\nInterprétation :")
print("- Écart faible entre validation et test -> le modèle généralise de")
print("  façon cohérente (pas de surapprentissage sur la validation via")
print("  l'optimisation Optuna).")
print("- Les deux scores restant proches du taux de base -> confirme, sur")
print("  des données totalement inédites, l'absence de signal exploitable")
print("  déjà établie (preuve n°8, la plus solide : généralisation testée")
print("  sur un ensemble jamais vu).")

## 3. RÉSULTATS DÉTAILLÉS SUR LE TEST (résultat officiel du projet)

In [ ]:
print("\n" + "=" * 70)
print("RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST (résultat officiel)")
print("=" * 70)

auc_pr_test = average_precision_score(y_test, proba_test)
auc_roc_test = roc_auc_score(y_test, proba_test)

print(f"AUC-PR (test) : {auc_pr_test:.4f}")
print(f"AUC-ROC (test) : {auc_roc_test:.4f}")
print(f"\nRapport de classification complet (seuil 0.5) :")
print(classification_report(y_test, pred_test, target_names=["Non-fraude", "Fraude"]))

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
print(f"Matrice de confusion -> VP={tp} FP={fp} FN={fn} VN={tn}")

## 4. COURBE PRECISION-RECALL SUR LE TEST

In [ ]:
precisions, rappels, seuils = precision_recall_curve(y_test, proba_test)

plt.figure(figsize=(8, 6))
plt.plot(rappels, precisions, color="#2E86AB", label=f"Modèle (AUC-PR={auc_pr_test:.4f})")
plt.axhline(y=y_test.mean(), color="#E63946", linestyle="--",
            label=f"Hasard (taux de base={y_test.mean():.4f})")
plt.xlabel("Rappel (Recall)")
plt.ylabel("Précision")
plt.title("Courbe Precision-Recall — Ensemble de test")
plt.legend()
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/courbe_pr_test.png", dpi=120)
#plt.close()
print(f"\nGraphique sauvegardé : courbe_pr_test.png")
print("Note : une courbe quasi confondue avec la ligne du hasard confirme")
print("visuellement, sur données de test, l'absence de pouvoir de")
print("discrimination du modèle.")

## 5. MATRICE DE CONFUSION VISUELLE

In [ ]:
plt.figure(figsize=(6, 5))
matrice = confusion_matrix(y_test, pred_test)
sns.heatmap(matrice, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-fraude", "Fraude"],
            yticklabels=["Non-fraude", "Fraude"])
plt.xlabel("Prédiction")
plt.ylabel("Réel")
plt.title("Matrice de confusion — Ensemble de test")
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/matrice_confusion_test.png", dpi=120)
#plt.close()
print("Graphique sauvegardé : matrice_confusion_test.png")

print("""
NOTE MÉTHODOLOGIQUE (à reprendre dans le rapport, section résultats finaux) :
L'évaluation finale a été réalisée sur l'ensemble de test (15% du dataset,
210 146 lignes), non utilisé lors de la comparaison des modèles ni lors de
l'optimisation des hyperparamètres, garantissant une estimation non biaisée
de la performance de généralisation. Le résultat obtenu (AUC-PR = {auc_pr:.4f})
est cohérent avec les scores de validation et confirme, sur des données
totalement inédites pour le modèle, l'absence de signal exploitable
identifiée lors du diagnostic préalable.
""".format(auc_pr=auc_pr_test))